# Bambara SLM: Stage 1 continued pretraining on Colab

**Before you start**
1. *Runtime → Change runtime type → GPU*. A100 is best; L4 is fine; T4 (free) works but is slow.
2. Upload `bambara_slm_colab.zip` (made on the Mac by `scripts/make_colab_bundle.sh`) to Google Drive, folder `MyDrive/bambara-SLM/`.
3. Run the cells in order. **If Colab disconnects, reconnect and run all cells again:** training resumes from the last checkpoint saved on Drive (every 50 steps).

One variant per run: first `with_nllb`, then (new session or after it finishes) `no_nllb`.

In [ ]:
VARIANT = "with_nllb"          # then "no_nllb" for the comparison run
DRIVE_DIR = "/content/drive/MyDrive/bambara-SLM"
BUNDLE = f"{DRIVE_DIR}/bambara_slm_colab.zip"
RUN_DIR = f"{DRIVE_DIR}/runs/cpt_{VARIANT}"      # checkpoints + results live on Drive (survive disconnects)

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
p = torch.cuda.get_device_properties(0)
cc = torch.cuda.get_device_capability(0)
print(torch.cuda.get_device_name(0), f"{p.total_memory/2**30:.0f} GB", "| compute capability", cc,
      "| native bf16:", cc[0] >= 8, "(T4 = no: training uses fp32 weights + fp16 mixed precision)")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.exists(BUNDLE), f"Upload the bundle to {BUNDLE}"
os.makedirs(RUN_DIR, exist_ok=True)

In [ ]:
!rm -rf /content/bambara-SLM && mkdir -p /content/bambara-SLM && cd /content/bambara-SLM && unzip -q "{BUNDLE}"
!cd /content/bambara-SLM && pip install -q -r requirements-gpu.txt && (pip install -q -e . --no-deps || echo 'editable install skipped: PYTHONPATH is used instead')
# Colab preinstalls a torchao version that peft rejects (ImportError); we do not use torchao
!pip uninstall -y -q torchao
!ls /content/bambara-SLM/data/cpt

Optional: add a Hugging Face token as a Colab secret named `HF_TOKEN` (key icon on the left) for faster model download. The model is public, so this is not required.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN secret (fine)")

## Train
Runs for 1–2 h (A100), 3–5 h (L4) or 8–12+ h (T4), roughly. It prints Bambara / French / English validation loss every 50 steps.
If the session drops, reconnect and **run all cells again**; it resumes.

In [ ]:
os.environ["PYTHONPATH"] = "/content/bambara-SLM/src"   # works even if `pip install -e .` is skipped
os.environ["BSLM_RUN_DIR"] = RUN_DIR
os.environ["BSLM_MAX_HOURS"] = "3.5"   # free Colab sessions end after ~4 h: stop cleanly with a checkpoint before that
os.environ["PYTHONUNBUFFERED"] = "1"
!cd /content/bambara-SLM && python -m bslm.train.cpt --config configs/train/cpt_{VARIANT}.yaml 2>&1 | tee -a "{RUN_DIR}/train.log"

## Result
Exit code 2 / `chosen_step: null` means no checkpoint kept French and English within +10 % perplexity: tell Claude (re-run with more replay).

In [ ]:
import json
print(json.dumps(json.load(open(f"{RUN_DIR}/selection.json")), indent=2))
print(json.dumps(json.load(open(f"{RUN_DIR}/samples.json")), ensure_ascii=False, indent=2))

## Bring the result to the Mac
The merged model is in `MyDrive/bambara-SLM/runs/cpt_<variant>/merged` (≈ 3.4 GB). Download that folder plus the
`.json` reports, either with Google Drive for desktop or with the cell below, which makes a single zip without the checkpoints.
Put it in `~/Documents/bambara-SLM/runs/` and tell Claude.

In [ ]:
!cd "{DRIVE_DIR}/runs" && zip -qr "cpt_{VARIANT}_results.zip" "cpt_{VARIANT}" -x "cpt_{VARIANT}/checkpoints/*" && ls -lh "cpt_{VARIANT}_results.zip"